In [1]:
import numpy as np
from pathlib import Path

# Define the data directory
data_dir = Path('/pscratch/sd/v/valer/atmo3/atmo3/data')

# The exact names of the files we just built scripts for
target_files = [
    'partition_functions.npz',
    'h2o_lines_1750GHz.npz',
    'mt_ckd_continuum.npz',
    'o2_coupled_lines_1750GHz.npz',
    'o2_uncoupled_lines_1750GHz.npz',
    'cia_tables_no_radiation_1750GHz.npz'
]

def load_all_data(directory, files):
    """Loads all arrays from a list of .npz files into a nested dictionary."""
    data_dict = {}
    for fname in files:
        fpath = directory / fname
        if fpath.exists():
            # Load the archive
            with np.load(fpath) as archive:
                # Extract all arrays into a standard Python dictionary
                data_dict[fname] = {key: archive[key] for key in archive.files}
            print(f"Loaded: {fname} ({len(data_dict[fname])} arrays)")
        else:
            print(f"WARNING: File not found - {fname}")
            
    return data_dict

# 1. Capture the BEFORE state
print("--- Loading BEFORE Data ---")
data_before = load_all_data(data_dir, target_files)

--- Loading BEFORE Data ---
Loaded: partition_functions.npz (4 arrays)
Loaded: h2o_lines_1750GHz.npz (7 arrays)
Loaded: mt_ckd_continuum.npz (4 arrays)
Loaded: o2_coupled_lines_1750GHz.npz (12 arrays)
Loaded: o2_uncoupled_lines_1750GHz.npz (6 arrays)
Loaded: cia_tables_no_radiation_1750GHz.npz (4 arrays)


In [2]:
# 2. Capture the AFTER state
print("--- Loading AFTER Data ---")
data_after = load_all_data(data_dir, target_files)

def compare_data(before, after):
    """Compares two nested data dictionaries for exact equality."""
    print("\n--- Commencing Comparison ---")
    all_match = True
    
    for fname in before.keys():
        if fname not in after:
            print(f"❌ ERROR: {fname} is missing in AFTER data!")
            all_match = False
            continue

        for arr_name in before[fname].keys():
            if arr_name not in after[fname]:
                print(f"❌ ERROR: Array '{arr_name}' missing in AFTER data for {fname}")
                all_match = False
                continue

            arr_before = before[fname][arr_name]
            arr_after = after[fname][arr_name]
            
            # Check if shapes match first
            if arr_before.shape != arr_after.shape:
                print(f"❌ SHAPE MISMATCH in {fname} -> '{arr_name}': Before {arr_before.shape} vs After {arr_after.shape}")
                all_match = False
                continue

            # Check if all numbers are mathematically identical
            # (np.allclose handles tiny floating point truncation differences)
            if not np.allclose(arr_before, arr_after, rtol=1e-10, atol=1e-12):
                print(f"❌ VALUE MISMATCH in {fname} -> '{arr_name}'")
                
                # Print a sample of the error to help debug
                diff = np.abs(arr_before - arr_after)
                max_diff = np.max(diff)
                print(f"   Max absolute difference: {max_diff}")
                
                all_match = False
            else:
                print(f"✅ Match: {fname} -> '{arr_name}'")

    if all_match:
        print("\n🎉 SUCCESS: All data files are mathematically identical! The refactor is perfect.")
    else:
        print("\n⚠️ WARNING: Differences found. Check the logs above.")

# 3. Execute the Comparison
compare_data(data_before, data_after)

--- Loading AFTER Data ---
Loaded: partition_functions.npz (4 arrays)
Loaded: h2o_lines_1750GHz.npz (7 arrays)
Loaded: mt_ckd_continuum.npz (4 arrays)
Loaded: o2_coupled_lines_1750GHz.npz (12 arrays)
Loaded: o2_uncoupled_lines_1750GHz.npz (6 arrays)
Loaded: cia_tables_no_radiation_1750GHz.npz (4 arrays)

--- Commencing Comparison ---
✅ Match: partition_functions.npz -> 'T_grid'
✅ Match: partition_functions.npz -> 'H2O'
✅ Match: partition_functions.npz -> 'O3'
✅ Match: partition_functions.npz -> 'O2'
✅ Match: h2o_lines_1750GHz.npz -> 'f0'
✅ Match: h2o_lines_1750GHz.npz -> 'S'
✅ Match: h2o_lines_1750GHz.npz -> 'ga'
✅ Match: h2o_lines_1750GHz.npz -> 'gs'
✅ Match: h2o_lines_1750GHz.npz -> 'n'
✅ Match: h2o_lines_1750GHz.npz -> 'E'
✅ Match: h2o_lines_1750GHz.npz -> 'd'
✅ Match: mt_ckd_continuum.npz -> 'nu'
✅ Match: mt_ckd_continuum.npz -> 'Cs'
✅ Match: mt_ckd_continuum.npz -> 'Cf'
✅ Match: mt_ckd_continuum.npz -> 'Texp'
✅ Match: o2_coupled_lines_1750GHz.npz -> 'f0'
✅ Match: o2_coupled_lines_